# 노트북 상태와 가격표 읽기

저장된 출력과 현재 커널 상태를 구분하고, 가격표의 행·열·인덱스·자료형·결측·중복·날짜 범위를 확인합니다.

## 1. 입력 파일을 읽어 현재 커널에 `prices`를 만든다

이 셀을 실행하지 않은 새 커널에는 `prices`가 없습니다. 아래에 예전 실행 결과가 보이더라도 현재 커널에 변수가 있다는 뜻은 아닙니다.

In [1]:
from pathlib import Path

import pandas as pd

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
csv_path = project_root / "data" / "kodex200-price-fixture.csv"

prices = pd.read_csv(csv_path, parse_dates=["Date"])
print(f"불러오기 완료: {len(prices)}행 × {len(prices.columns)}열")

불러오기 완료: 12행 × 7열


## 2. 표의 행과 열을 눈으로 먼저 확인한다

이 셀을 새 커널에서 먼저 실행해 `NameError`를 확인한 뒤, 위 셀을 실행하고 다시 시도합니다.

In [2]:
display(prices.head(4))

,Date,Open,High,Low,Close,Volume,Change
0,2024-01-02,34157,34676,34100,34541,7815352,0.006088
1,2024-01-03,34147,34147,33583,33583,8532058,-0.027735
2,2024-01-04,33325,33555,33199,33287,9353614,-0.008814
3,2024-01-05,33276,33382,33136,33194,7018655,-0.002794


## 3. 표 구조 점검표를 읽는다

아래 함수는 이 12행 발췌본의 기대 상태를 검사합니다. 날짜 계산의 선행 조건이 없으면 확인 불가로 표시합니다.


In [3]:
def table_check(table):
    expected_columns = ["Date", "Open", "High", "Low", "Close", "Volume", "Change"]
    dates = table["Date"] if "Date" in table.columns else None
    date_type_ok = dates is not None and pd.api.types.is_datetime64_any_dtype(dates)
    dates_ready = date_type_ok and len(dates) > 0 and not dates.isna().any()
    missing_count = int(table.isna().sum().sum())
    duplicate_count = int(dates.duplicated().sum()) if dates is not None else None
    basic_index = isinstance(table.index, pd.RangeIndex) and table.index.equals(pd.RangeIndex(len(table)))
    date_range = "확인 불가"
    range_ok = None
    ascending = None
    if dates_ready:
        date_range = f"{dates.min().date()} ~ {dates.max().date()}"
        range_ok = dates.min() == pd.Timestamp("2024-01-02") and dates.max() == pd.Timestamp("2024-01-17")
        ascending = bool(dates.is_monotonic_increasing)

    checks = pd.DataFrame(
        [
            ("행 × 열", f"{len(table)} × {len(table.columns)}", len(table) == 12 and len(table.columns) == 7),
            ("열 이름", ", ".join(table.columns), list(table.columns) == expected_columns),
            ("인덱스", type(table.index).__name__, basic_index),
            ("Date 자료형", str(dates.dtype) if dates is not None else "열 없음", date_type_ok if dates is not None else None),
            ("전체 결측", f"{missing_count}개", missing_count == 0),
            ("중복 날짜", f"{duplicate_count}개" if duplicate_count is not None else "확인 불가", duplicate_count == 0 if duplicate_count is not None else None),
            ("날짜 정렬", "확인 불가" if ascending is None else ("오름차순" if ascending else "섞임"), ascending),
            ("날짜 범위", date_range, range_ok),
        ],
        columns=["확인 항목", "결과", "판정"],
    )
    checks["판정"] = checks["판정"].map(lambda value: "확인 불가" if value is None else ("통과" if value else "확인 필요"))
    display(checks)
    if (checks["판정"] != "통과").any():
        raise RuntimeError("표 구조 점검에서 확인할 항목이 있습니다. 위 점검표를 읽으세요.")
    print("표 구조 점검 통과")

table_check(prices)


,확인 항목,결과,판정
0,행 × 열,12 × 7,통과
1,열 이름,"Date, Open, High, Low, Close, Volume, Change",통과
2,인덱스,RangeIndex,통과
3,Date 자료형,datetime64[ns],통과
4,전체 결측,0개,통과
5,중복 날짜,0개,통과
6,날짜 정렬,오름차순,통과
7,날짜 범위,2024-01-02 ~ 2024-01-17,통과


표 구조 점검 통과


## 마무리

커널을 다시 시작하고 모든 셀을 위에서부터 실행했을 때 마지막에 **표 구조 점검 통과**가 나오면, 저장된 출력이 아니라 현재 실행으로 표를 다시 만든 것입니다.